In [ ]:
import csv
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

RUNNER_CONFIG = json.loads("{\"det_threshold\": 0.965, \"det_tta_mode\": \"dihedral8\", \"edge_max_distance_um\": 10.0, \"edge_min_threshold\": 0.35, \"edge_threshold\": 0.48, \"edge_topk_children\": 2, \"edge_topk_parents\": 2, \"gap_closing\": true, \"gap_closing_max_added_absolute\": 1000, \"gap_closing_max_added_fraction\": 0.02, \"gap_closing_max_total_distance_um\": 10.0, \"gap_closing_reuse_existing_um\": 2.5, \"ilp_appearance_weight\": 0.0, \"ilp_disappearance_weight\": 1.8, \"ilp_division_weight\": 0.5, \"metric_probe\": false, \"metric_probe_forks\": 5, \"metric_probe_max_components\": 1400, \"pool_kernel_um\": 3.0}")
COMPETITION_ROOT_CANDIDATES = [
    Path('/kaggle/input/competitions/biohub-cell-tracking-during-development'),
    Path('/kaggle/input/biohub-cell-tracking-during-development'),
]
REQUIRED_IMPORTS = (
    'tracksdata', 'zarr', 'numcodecs', 'geff', 'pyscipopt',
    'ilpy', 'rustworkx', 'polars', 'dask',
)
OFFLINE_PACKAGES = (
    'tracksdata', 'zarr==3.2.1', 'numcodecs==0.15.1',
    'donfig==0.8.1.post1', 'geff==1.2.0.1.1', 'geff-spec==1.1.1',
    'pyscipopt==6.2.1', 'ilpy==0.6.0', 'rustworkx==0.18.0',
    'polars==1.42.0', 'polars-runtime-32==1.42.0',
    'bidict==0.23.1', 'imagecodecs==2026.6.26',
    'dask==2026.6.0', 'blosc2==4.7.0', 'pyarrow==24.0.0',
    'sqlalchemy==2.0.51',
)

def find_support_pack():
    for manifest in sorted(Path('/kaggle/input').glob('**/ARTIFACT_MANIFEST.json')):
        root = manifest.parent
        if (root / 'repo' / 'src' / 'biohub_tracking').is_dir() and (
            root / 'weights' / 'unet_transformer' / 'split_0' / 'edge_predictor_best.pth'
        ).is_file():
            return root
    raise FileNotFoundError('public Biohub tracking support pack was not attached')

def require_offline_dependencies(support_pack):
    missing = [name for name in REQUIRED_IMPORTS if importlib.util.find_spec(name) is None]
    if not missing:
        return
    wheels = support_pack / 'wheels'
    if not wheels.is_dir():
        raise FileNotFoundError('support pack wheels directory is missing: ' + str(wheels))
    command = [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', '--no-deps',
               '--find-links', str(wheels), *OFFLINE_PACKAGES]
    completed = subprocess.run(command, capture_output=True, text=True, check=False)
    if completed.returncode != 0:
        detail = (completed.stdout + '\n' + completed.stderr).strip()[-4000:]
        raise RuntimeError('offline support-pack dependency installation failed: ' + detail)
    importlib.invalidate_caches()

def resolve_competition_root():
    override = os.environ.get('BIOHUB_COMPETITION_DIR', '').strip()
    candidates = [Path(override)] if override else COMPETITION_ROOT_CANDIDATES
    for candidate in candidates:
        if (candidate / 'test').is_dir():
            return candidate
    raise FileNotFoundError('competition test directory not found: ' + ', '.join(map(str, candidates)))

def prepare_batch_view(test_dir, working_dir):
    batch_dir = working_dir / 'test_batch'
    if batch_dir.exists():
        shutil.rmtree(batch_dir)
    batch_dir.mkdir(parents=True, exist_ok=True)
    names = []
    for source in sorted(test_dir.glob('*.zarr')):
        name = source.stem
        (batch_dir / (name + '.zarr')).symlink_to(source, target_is_directory=True)
        names.append(name)
    if not names:
        raise FileNotFoundError('no .zarr datasets found in ' + str(test_dir))
    split_path = batch_dir / 'dataset_splits.json'
    split_path.write_text(json.dumps([{'test': names}]), encoding='utf-8')
    return batch_dir, names, split_path

def load_graph(path, td):
    result = td.graph.IndexedRXGraph.from_geff(path)
    return result[0] if isinstance(result, tuple) else result


def _find_gap_closing_package():
    matches = sorted(Path('/kaggle/input').rglob('postprocess/last_call.py'))
    if not matches:
        raise FileNotFoundError('legal gap-closing package was not attached')
    return matches[0].parent.parent


def _apply_gap_closing(rows, data_root):
    if not RUNNER_CONFIG['gap_closing']:
        return rows
    package_root = _find_gap_closing_package()
    sys.path.insert(0, str(package_root))
    import pandas as pd
    from postprocess.last_call import GapClosingConfig, postprocess_submission

    columns = ['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']
    frame = pd.DataFrame(rows, columns=columns)
    config = GapClosingConfig(
        max_total_distance_um=RUNNER_CONFIG['gap_closing_max_total_distance_um'],
        reuse_existing_um=RUNNER_CONFIG['gap_closing_reuse_existing_um'],
        max_added_fraction=RUNNER_CONFIG['gap_closing_max_added_fraction'],
        max_added_absolute=RUNNER_CONFIG['gap_closing_max_added_absolute'],
        synthetic_mode='intensity_centroid',
        reuse_existing=True,
        min_track_length=0,
    )
    clean, report = postprocess_submission(frame, data_root, config)
    print('GAP_CLOSING_REPORT:' + report.to_json(orient='records'), flush=True)
    return clean.to_dict('records')
def write_submission(graph_dir, dataset_names, output_path, td):
    columns = ['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']
    rows = []
    for dataset in dataset_names:
        graph_path = graph_dir / (dataset + '.geff')
        if not graph_path.exists():
            raise FileNotFoundError('prediction graph missing: ' + str(graph_path))
        graph = load_graph(graph_path, td)
        nodes = list(graph.node_attrs().iter_rows(named=True))
        edges = list(graph.edge_attrs().iter_rows(named=True))
        node_ids = {int(row['node_id']) for row in nodes}
        if not node_ids:
            raise ValueError(dataset + ': learned model produced no nodes')
        for row in sorted(nodes, key=lambda item: int(item['node_id'])):
            rows.append({'id': -1, 'dataset': dataset, 'row_type': 'node',
                'node_id': int(row['node_id']), 't': int(row['t']),
                'z': max(0, int(round(float(row['z'])))),
                'y': max(0, int(round(float(row['y'])))),
                'x': max(0, int(round(float(row['x'])))),
                'source_id': -1, 'target_id': -1})
        for row in edges:
            source_id, target_id = int(row['source_id']), int(row['target_id'])
            if source_id not in node_ids or target_id not in node_ids:
                raise ValueError(dataset + ': dangling edge')
            rows.append({'id': -1, 'dataset': dataset, 'row_type': 'edge',
                'node_id': -1, 't': -1, 'z': -1, 'y': -1, 'x': -1,
                'source_id': source_id, 'target_id': target_id})
    if not rows:
        raise ValueError('submission contains no learned nodes')
    rows = _apply_gap_closing(rows, competition_root / 'test')
    if RUNNER_CONFIG['metric_probe']:
        rows = _augment_metric_probe(rows, dataset_names)
    for row_id, row in enumerate(rows):
        row['id'] = row_id
    with output_path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)

cycle_started = time.monotonic()
timings = {}
stage_started = time.monotonic()
support_pack = find_support_pack()
require_offline_dependencies(support_pack)
timings['dependency_setup_seconds'] = round(time.monotonic() - stage_started, 3)
stage_started = time.monotonic()
competition_root = resolve_competition_root()
working_dir = Path(os.environ.get('BIOHUB_WORKING_DIR', '/kaggle/working'))
working_dir.mkdir(parents=True, exist_ok=True)
runtime_repo = working_dir / 'support_repo'
if runtime_repo.exists():
    shutil.rmtree(runtime_repo)
shutil.copytree(support_pack / 'repo', runtime_repo)

def _patch_support_predictor(predict_script):
    source = predict_script.read_text(encoding='utf-8')
    if 'edge_topk_parents' in source:
        return
    config_marker = '    threshold: float = 0.5\n'
    if config_marker not in source:
        raise RuntimeError('support-pack predictor config marker not found')
    source = source.replace(
        config_marker,
        config_marker
       + '    edge_min_threshold: float = 0.0\n'
       + '    edge_topk_parents: int = 0\n'
        + '    edge_topk_children: int = 0\n'
       + '    edge_max_distance_um: float = 0.0\n',
        1,
    )
    candidate_block = '''            candidates = sorted(\n                [\n                    (probs[i, j], i, j)\n                    for i in range(n_src)\n                    for j in range(n_tgt)\n                    if probs[i, j] > cfg.threshold\n                ],\n                reverse=True,\n            )\n'''
    candidate_replacement = '''            candidate_pairs = {\n                (i, j)\n                for i in range(n_src)\n                for j in range(n_tgt)\n                if probs[i, j] > cfg.threshold\n            }\n            if cfg.edge_topk_parents > 0 and cfg.edge_min_threshold > 0.0:\n                for j in range(n_tgt):\n                    parents = np.argsort(probs[:, j])[::-1][:cfg.edge_topk_parents]\n                    for i in parents:\n                        if probs[i, j] >= cfg.edge_min_threshold:\n                            candidate_pairs.add((int(i), j))\n            if cfg.edge_topk_children > 0 and cfg.edge_min_threshold > 0.0:\n                for i in range(n_src):\n                    children = np.argsort(probs[i, :])[::-1][:cfg.edge_topk_children]\n                    for j in children:\n                        if probs[i, j] >= cfg.edge_min_threshold:\n                            candidate_pairs.add((i, int(j)))\n            candidates = sorted(\n                [(probs[i, j], i, j) for i, j in candidate_pairs],\n                reverse=True,\n            )\n'''
    if candidate_block not in source:
        raise RuntimeError('support-pack predictor edge candidate block not found')
    source = source.replace(candidate_block, candidate_replacement, 1)
    distance_block = '''                dist = float(np.linalg.norm(\n                    coords_so_far[gi, 1:].astype(np.float32)\n                    - coords_so_far[gj, 1:].astype(np.float32)\n                ))\n'''
    distance_replacement = '''                dist = float(np.linalg.norm(\n                    (coords_so_far[gi, 1:].astype(np.float32)\n                     - coords_so_far[gj, 1:].astype(np.float32))\n                    * np.asarray(voxel_size, dtype=np.float32)\n                ))\n                if cfg.edge_max_distance_um > 0.0 and dist > cfg.edge_max_distance_um:\n                    continue\n'''
    if distance_block not in source:
        raise RuntimeError('support-pack predictor distance block not found')
    source = source.replace(distance_block, distance_replacement, 1)
    tta_config_marker = '    edge_max_distance_um: float = 0.0\n'
    if tta_config_marker not in source:
        raise RuntimeError('support-pack predictor TTA config marker not found')
    source = source.replace(
        tta_config_marker,
        tta_config_marker + '    det_tta_mode: str = "flip4"\n',
        1,
    )
    tta_block = '''        # Detection TTA: original + flip-x + flip-y + flip-xy, average logits.\n        # TTA: flip along Y (-2) and X (-1) only.  Z is excluded because\n        # the data is highly anisotropic (Z resolution ~4x coarser than XY),\n        # so Z-flips would produce out-of-distribution inputs.\n        if cfg.det_tta:\n            tta_flips = [(-1,), (-2,), (-2, -1)]\n            for dims in tta_flips:\n                imgs_flip = imgs.flip(dims)\n                _, det_flip = model.encode(imgs_flip)\n                for f in range(W):\n                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)\n                del imgs_flip, det_flip\n            for f in range(W):\n                det_logits[f] = det_logits[f] / 4\n'''
    tta_replacement = '''        # Detection TTA: use the public lead's XY dihedral group when requested.\n        if cfg.det_tta:\n            if cfg.det_tta_mode == "dihedral8":\n                tta_transforms = [\n                    (lambda value: value.flip((-1,)), lambda value: value.flip((-1,))),\n                    (lambda value: value.flip((-2,)), lambda value: value.flip((-2,))),\n                    (lambda value: value.flip((-2, -1)), lambda value: value.flip((-2, -1))),\n                    (lambda value: torch.rot90(value, 1, dims=(-2, -1)),\n                     lambda value: torch.rot90(value, -1, dims=(-2, -1))),\n                    (lambda value: torch.rot90(value, -1, dims=(-2, -1)),\n                     lambda value: torch.rot90(value, 1, dims=(-2, -1))),\n                    (lambda value: value.transpose(-1, -2),\n                     lambda value: value.transpose(-1, -2)),\n                    (lambda value: torch.rot90(value, 1, dims=(-2, -1)).transpose(-1, -2),\n                     lambda value: torch.rot90(value.transpose(-1, -2), -1, dims=(-2, -1))),\n                ]\n            else:\n                tta_transforms = [\n                    (lambda value: value.flip((-1,)), lambda value: value.flip((-1,))),\n                    (lambda value: value.flip((-2,)), lambda value: value.flip((-2,))),\n                    (lambda value: value.flip((-2, -1)), lambda value: value.flip((-2, -1))),\n                ]\n            for forward, inverse in tta_transforms:\n                imgs_flip = forward(imgs)\n                _, det_flip = model.encode(imgs_flip)\n                for f in range(W):\n                    det_logits[f] = det_logits[f] + inverse(det_flip[f])\n                del imgs_flip, det_flip\n            for f in range(W):\n                det_logits[f] = det_logits[f] / (len(tta_transforms) + 1)\n'''
    if tta_block not in source:
        raise RuntimeError('support-pack predictor TTA block not found')
    source = source.replace(tta_block, tta_replacement, 1)
    predict_script.write_text(source, encoding='utf-8')

_patch_support_predictor(runtime_repo / 'scripts' / 'predict_unet_transformer.py')
sys.path.insert(0, str(runtime_repo / 'src'))
sys.path.insert(0, str(runtime_repo / 'scripts'))
os.environ['USER'] = 'biohub_kaggle_support'
batch_dir, dataset_names, split_path = prepare_batch_view(competition_root / 'test', working_dir)
timings['input_setup_seconds'] = round(time.monotonic() - stage_started, 3)
from predict_unet_transformer import PredictConfig, predict
import tracksdata as td
weights = support_pack / 'weights' / 'unet_transformer' / 'split_0' / 'edge_predictor_best.pth'
config = PredictConfig(
    det_threshold=RUNNER_CONFIG['det_threshold'],
    threshold=RUNNER_CONFIG['edge_threshold'],
    edge_min_threshold=RUNNER_CONFIG['edge_min_threshold'],
    edge_topk_parents=RUNNER_CONFIG['edge_topk_parents'],
    edge_topk_children=RUNNER_CONFIG['edge_topk_children'],
    edge_max_distance_um=RUNNER_CONFIG['edge_max_distance_um'],
    det_tta_mode=RUNNER_CONFIG['det_tta_mode'],
    pool_kernel_um=RUNNER_CONFIG['pool_kernel_um'],
    use_ilp=True,
    ilp_edge_weight=-1.0,
    ilp_appearance_weight=RUNNER_CONFIG['ilp_appearance_weight'],
    ilp_disappearance_weight=RUNNER_CONFIG['ilp_disappearance_weight'],
    ilp_division_weight=RUNNER_CONFIG['ilp_division_weight'],
)
stage_started = time.monotonic()
predict(
    data_dir=batch_dir, fold=0, splits_file=split_path, weights_path=weights,
    cfg=config, method='unet_transformer', unet_batch_size=4,
)
timings['prediction_seconds'] = round(time.monotonic() - stage_started, 3)
prediction_dir = runtime_repo / 'predictions' / 'biohub_kaggle_support' / 'unet_transformer' / 'split_0'
stage_started = time.monotonic()
write_submission(prediction_dir, dataset_names, working_dir / 'submission.csv', td)
timings['serialization_postprocess_seconds'] = round(time.monotonic() - stage_started, 3)
timings['total_seconds'] = round(time.monotonic() - cycle_started, 3)
print(json.dumps({'datasets': len(dataset_names), 'submission': str(working_dir / 'submission.csv'), 'timings_seconds': timings}, sort_keys=True), flush=True)
